# Symmetric functions: Fock vectors, alpha operators and characters

Each section below was a separate notebook. The sections share one kernel, so run a section from its own first cell.

## Alpha operators on Fock vectors

In [12]:
%display latex
import code as pc
import numpy as np
Partitions.options.latex='list'

### to_fock ($\lambda$):

Takes a partition $$\lambda = \lambda_1 + \cdots + \lambda_\ell$$ of $n$, returns the fock vector $\mathbf{v}$ which satisfies $$\mathbf{v}_i = \lambda_i - i.$$

*(Note that $i+1$ is needed everywhere due to Python's 0-based indexing.)*

### to_partition ($\mathbf{v}$):

Takes a fock vector $\mathbf{v}$ and returns the partition $\lambda$ satisfying the above condition.

In [13]:
# Given a partition 
def to_fock(partition):
    return tuple([(x-(i+1)) for i, x in enumerate(partition)])

def to_fock_singleton(partition):
    v = to_fock(partition)
    return([(1, v)])

def to_partition(fock_vector):
    return Partition([(x+(i+1)) for i, x in enumerate(fock_vector)])

### num_swaps (L):

Takes a list $L$, and produces a sorted list $L'$ that is obtained from $L$ by using only transpositions. Also keeps track of $s$, the number of transpositions needed, and produces the following sign:
$$
\mathrm{sign}(s) = \begin{cases}-1 & \text{if } s \text{ is even} \\ +1 & \text{else} \end{cases}
$$

In [31]:
# Take a list, sort it into weakly decreasing order using only
# transpositions, and count the number of transpositions needed.
def num_swaps(L):
    Lp = copy(L)
    swapcount = 0
    for j in range(len(Lp)):
        for i in range(1, len(Lp)-j):
            if Lp[i-1] > Lp[i]:
                swapcount += 1
                Lp[i-1], Lp[i] = Lp[i], Lp[i-1]
    sign = 1 if swapcount % 2 == 0 else -1
    Lpp = list(Lp)
    Lpp.reverse()
    return sign, tuple(Lpp)

### $\alpha_k(\mathbf{v}_\lambda)$

Given a fock vector $\mathbf{v}_\lambda = [v_1, v_2, \cdots, v_\ell]$, first computes the application
$$
\begin{align*}
v_1 + \mathbf{k} &\wedge v_2     &\wedge \cdots &\wedge v_\ell \\
v_1     &\wedge v_2 + \mathbf{k} &\wedge \cdots &\wedge v_\ell \\
v_1 &\wedge v_2 &\wedge \cdots &\wedge v_\ell + \mathbf{k}
\end{align*}
$$

We do this by stacking $\mathbf{v}$ as the rows of a matrix, and subtracting the scalar matrix $kI$.

We then proceed to wedge each summand above back into weakly decreasing order. 

- Noting that any summand with a repeated entry will wedge to zero, we delete it.

- We use the *num_swaps* function to sort each vector using transpositions, tracking the sign of the transpositions.


In [49]:
def vacuum(n):
    return([-i-1 for i in range(n)])

def partial_vacuum(n, k):
    l1 = vector(vaccum(n))
    l2 = vector(vaccum(k) + [ZZ(0)]*(n-k))
    return((l1 - l2))

def alpha(k, v_lambda):
    
    # Length of fock vector
    n = len(v_lambda)
                
    v_lambda
    # Stack n copies of v_lambda up as the rows of an nxn matrix
    V = Matrix([ v_lambda for i in range(n) ])
    
    # A scalar matrix for k
    kI = identity_matrix(n) * k
    
    # Subtract k off of the diagonal
    M = V - kI
    
    # Sorted vectors, with the (signs of the) number of transpositions
    # Throw away vectors with repeated entries, since these
    # wedge to zero.
    L = [num_swaps(R) for R in M.rows() if len(R) == len(set(R))]

    return(L)

def distribute(lin_comb):
    outs = []
    for vs in lin_comb:
        c0 = vs[0]
        for v in vs[1]:
            ci, vi = v
            outs.append((ci * c0, vi ))
    return(outs)

def collect_terms(list_of_vs):
    D = dict()
    for i, x in enumerate(list_of_vs):
        c_i, v = x
        D[v] = c_i if v not in D.keys() else D[v] + c_i
    return([(D[i], i) for i in D if D[i] != 0])

def alpha_bar(k, ls):
    lin_comb = [(ci, alpha(k, vi)) for ci, vi in ls]
    reduced = distribute(lin_comb)
    more_reduced = collect_terms(reduced)
    return(more_reduced)

In [42]:
v_l = to_fock_singleton((3,1,1))
show(v_l)

[(1, (2, -1, -2))]

In [48]:
a2 = alpha(-2, v_l[0][1])
a2

[(-1, (4, -1, -2)), (-1, (2, 1, -2)), (1, (2, 0, -1))]

In [44]:
a2 = alpha(0, v_l[0][1])
a2

[(-1, (2, -1, -2)), (-1, (2, -1, -2)), (-1, (2, -1, -2))]

In [52]:
alpha_bar(-2, v_l)

[(-1, (4, -1, -2)), (-1, (2, 1, -2)), (1, (2, 0, -1))]

In [53]:
a3 = alpha_bar(3, a2)
a3

[(-1, (4, -2, -4)),
 (1, (4, -1, -5)),
 (1, (2, 1, -5)),
 (1, (2, -1, -3)),
 (-1, (2, 0, -4))]

### Notes from Monday Feb 24th

> DZG: From conversation with Richard earlier!

Note that $\alpha_k$ only works on a single basis fock vector $\mathbf{v}_\lambda$ -- in general, it should operate on a formal linear combination of fock vectors $\mathbf{v} = \sum_{i=1}^n c_i \mathbf{v}_i$.

How to do this? Idea: represent $\mathbf v = \sum_{i=1}^n c_i \mathbf{v}_i$ as a list of pairs 
$$
\mathbf v = [(c_1, \mathbf{v}_1), (c_2, \mathbf{v}_2), \cdots, (c_n, \mathbf{v}_n)]
$$

It's then easy to set
$$
\alpha_k(\mathbf v) = 
[(c_1, \alpha_k(\mathbf{v}_1) ), (c_2, \alpha_k(\mathbf{v}_2)), \cdots, (c_n, \alpha_k(\mathbf{v}_n))]
$$

Note that in each pair, the second entry is now *itself* a list of pairs. In gory detail, we have
$$
\alpha_k(\mathbf v) = 
[~(c_1, ~[ (d_{11}, \mathbf{v}_{11}), (d_{12}, \mathbf{v}_{12}), \cdots ]~ ), ~\cdots~, (c_n,  ~[ (d_{n1}, \mathbf{v}_{n1}), (d_{n2}, \mathbf{v}_{n2}), \cdots ]~ )~]
$$

In [ ]:
var('z')
sun = 1/(e^(z/2)-e^(-z/2))
def pk(k, partition):
    fock_vector = to_fock(partition)
    if k%2==0:
        ck = 0
    if k%2==1:
        ck = (sun.taylor(z,0,k+1)).coefficient(z^k)/k.factorial()
    return(ck + sum((x+1/2)^k-(-i+1/2)^k for i,x in enumerate(fock_vector))/k.factorial())
print(pk(5,(3,2,1,1)))

In [ ]:
def chi(representation, cycle_type):
    for i in range(size(representation)-size(cycle_type)):
        cycle_type.append(1)
    fock_vector = to_fock(representation)
    for i in range(len(cycle_type)):
        fock_vector = alpha(cycle_type[i], fock_vector)
    return(fock_vector[1])

In [ ]:
def hook_lengths(partition):
    return(list of hook lengths)

In [ ]:
def dim(partition):
    return(size(partition)!/prod(hook_lengths(partition)))

In [ ]:
def conj_class_size(partition):
    return(size(partition)!/(prod(partition)*prod(numbers of times i appears!)))

In [ ]:
def eff(representation, cycle_type):
    return(chi(representation, cycle_type)*conj_class_size(cycle_type)/dim(partition))

In [ ]:
def pillowcase_weight(N,partition):
    (compute sign somehow)
    return sign*product(hook_lengths s.t. hook_length%N = pm 1)/prod(hook_lengths s.t. hook_length%N=0)^2

In [ ]:
def quotient_partitions(N,partition):

In [ ]:
def core_partition(N,partition):

## Alpha operators on Fock vectors (v3)

In [1]:
%display latex
import code as pc
import numpy as np
import itertools
Partitions.options.latex='list'

In [2]:
# Given a partition 
def to_fock(partition):
    return tuple([(x-(i+1)) for i, x in enumerate(partition)])

def to_fock_singleton(partition):
    v = to_fock(partition)
    return([(1, v)])

def to_partition(fock_vector):
    return Partition([(x+(i+1)) for i, x in enumerate(fock_vector)])

# Take a list, sort it into weakly decreasing order using only
# transpositions, and count the number of transpositions needed.
def num_swaps(L):
    Lp = copy(L)
    swapcount = 0
    for j in range(len(Lp)):
        for i in range(1, len(Lp)-j):
            if Lp[i-1] > Lp[i]:
                swapcount += 1
                Lp[i-1], Lp[i] = Lp[i], Lp[i-1]
    sign = 1 if swapcount % 2 == 0 else -1
    Lpp = list(Lp)
    Lpp.reverse()
    return sign, tuple(Lpp)

def alpha(k, v_lambda):
    
    # Length of fock vector
    n = len(v_lambda)
    
    # Stack n copies of v_lambda up as the rows of an nxn matrix
    V = Matrix([ v_lambda for i in range(n) ])
    
    # A scalar matrix for k
    kI = identity_matrix(n) * k
    
    # Subtract k off of the diagonal
    M = V - kI
    
    # Sorted vectors, with the (signs of the) number of transpositions
    # Throw away vectors with repeated entries, since these
    # wedge to zero.
    L = [num_swaps(R) for R in M.rows() if len(R) == len(set(R))]

    return(L)

def distribute(lin_comb):
    outs = []
    for vs in lin_comb:
        c0 = vs[0]
        for v in vs[1]:
            ci, vi = v
            outs.append((ci * c0, vi ))
    return(outs)

def collect_terms(list_of_vs):
    D = dict()
    for i, x in enumerate(list_of_vs):
        c_i, v = x
        D[v] = c_i if v not in D.keys() else D[v] + c_i
    return([(D[i], i) for i in D if D[i] != 0])

def alpha_bar(k, ls):
    lin_comb = [(ci, alpha(k, vi)) for ci, vi in ls]
    reduced = distribute(lin_comb)
    reduced_unique = collect_terms(reduced)
    return(reduced_unique)

In [3]:
v_l1 = to_fock((3,1,1))
v_l2 = to_fock_singleton((3,1,1))
show(v_l1)
show(v_l2)

(2, -1, -2)

[(1, (2, -1, -2))]

In [4]:
show(alpha(2, v_l1))
show(alpha_bar(2, v_l2))

[(-1, (0, -1, -2)), (1, (2, -2, -3)), (-1, (2, -1, -4))]

[(-1, (0, -1, -2)), (1, (2, -2, -3)), (-1, (2, -1, -4))]

## Character evaluation with alpha operators

In [91]:
%display latex
import code as pc
import numpy as np
Partitions.options.latex='list'

# Given a partition 
def to_fock(partition):
    return vector([(x-(i+1)) for i, x in enumerate(partition)])

def to_partition(fock_vector):
    return Partition([(x+(i+1)) for i, x in enumerate(fock_vector)])


# Take a list, sort it into weakly decreasing order using only
# transpositions, and count the number of transpositions needed.
def num_swaps(L):
    Lp = copy(L)
    swapcount = 0
    for j in range(len(Lp)):
        for i in range(1, len(Lp)-j):
            if Lp[i-1] > Lp[i]:
                swapcount += 1
                Lp[i-1], Lp[i] = Lp[i], Lp[i-1]
    sign = -1 if swapcount % 2 == 0 else 1
    Lpp = list(Lp)
    Lpp.reverse()
    return vector(Lpp), sign

def do_mult(l):
    (v, s) = num_swaps(l)
    return(s*v)

In [92]:
def alpha(k, v_lambda):
    
    # Length of fock vector
    n = len(v_lambda)
    
    # Stack n copies of v_lambda up as the rows of an nxn matrix
    V = Matrix([ v_lambda for i in range(n) ])
    
    # A scalar matrix for k
    kI = identity_matrix(n) * k
    
    # Subtract k off of the diagonal
    M = V - kI
    
    # Sorted vectors, with the (signs of the) number of transpositions
    # Throw away vectors with repeated entries, since these
    # wedge to zero.
    L = [do_mult(R) for R in M.rows() if len(R) == len(set(R))]
    return(sum(L))

def alphap(k, v_lambda):
    
    # Length of fock vector
    n = len(v_lambda)
    
    # Stack n copies of v_lambda up as the rows of an nxn matrix
    V = Matrix([ v_lambda for i in range(n) ])
    
    # A scalar matrix for k
    kI = identity_matrix(n) * k
    
    # Subtract k off of the diagonal
    M = V - kI
    
    # Sorted vectors, with the (signs of the) number of transpositions
    # Throw away vectors with repeated entries, since these
    # wedge to zero.
    L = [num_swaps(R) for R in M.rows() if len(R) == len(set(R))]
    
    match_out1 = [ l[0] for l in L ]
    match_out2 = [ l[1] for l in L ]
    return([ match_out1, match_out2 ])

In [12]:
l = [3,1,1]
v_l = to_fock(l)
show(l)

[3, 1, 1]

In [13]:
A = alpha(1, v_l)
A

(3, -2, -5)

In [121]:
parts = Partitions(5).list()
n = 5
m = len(parts)

M = identity_matrix(ZZ, m)

def pad_to_n(v):
    vp = (list(v) + m * [0])[:m]
    return(vector(vp))

for i,p in enumerate(parts):
    a = pad_to_n(to_fock(p))
    M.set_column(i, a)
    show(p, "_____", a)
M

[5] '_____' (4, 0, 0, 0, 0, 0, 0)

[4, 1] '_____' (3, -1, 0, 0, 0, 0, 0)

[3, 2] '_____' (2, 0, 0, 0, 0, 0, 0)

[3, 1, 1] '_____' (2, -1, -2, 0, 0, 0, 0)

[2, 2, 1] '_____' (1, 0, -2, 0, 0, 0, 0)

[2, 1, 1, 1] '_____' (1, -1, -2, -3, 0, 0, 0)

[1, 1, 1, 1, 1] '_____' (0, -1, -2, -3, -4, 0, 0)

[ 4  3  2  2  1  1  0]
[ 0 -1  0 -1  0 -1 -1]
[ 0  0  0 -2 -2 -2 -2]
[ 0  0  0  0  0 -3 -3]
[ 0  0  0  0  0  0 -4]
[ 0  0  0  0  0  0  0]
[ 0  0  0  0  0  0  0]

In [75]:
len(parts)

7

In [81]:
out = alpha(-2, alpha(-3, v_l))
show(out)
show(pad_to_n(out))

(-5, 0, 3)

(-5, 0, 3, 0, 0, 0, 0)

In [84]:
v = pad_to_n(out)
w = M.T*v
w

(-20, -15, -10, -16, -11, -11, -6)

In [86]:
w

(-20, -15, -10, -16, -11, -11, -6)

In [47]:
p.size()

5

In [48]:
n = p.size()
ZZ^n

Ambient free module of rank 5 over the principal ideal domain Integer Ring

In [88]:
lll = Partition([3,1,1])
to_fock(lll)

(2, -1, -2)

In [94]:
alphap(-3, to_fock(lll))

[[(5, -1, -2), (2, 1, -1)], [1, -1]]

In [144]:
w1 = alpha(-3, to_fock(lll))
w = alpha(-2, w1)
w

(-5, 0, 3)

In [145]:
wp = padout(w)
wp

(-5, 0, 3, -4, -5, -6, -7)

In [148]:
M*wp

(-33, 27, 50, 83, 113, 144, 168)

In [108]:
to_fock(pad_to_n(wp))

(2, -4, -4, -4, -5, -6, -7)

In [135]:
def padout(partition):
    v0 = zero_vector(7)
    for i in range(7):
        if i < len(partition):
            v0[i] = partition[i]
        else:
            v0[i] = -i-1
    return(v0)

In [136]:
padout([2,-1,-2])

(2, -1, -2, -4, -5, -6, -7)

In [138]:
M = identity_matrix(ZZ, 7)
parts = Partitions(5)
for i,p in enumerate(parts):
    a = padout(to_fock(p))
    show(a)
    M.set_column(i, a)
M

(4, -2, -3, -4, -5, -6, -7)

(3, -1, -3, -4, -5, -6, -7)

(2, 0, -3, -4, -5, -6, -7)

(2, -1, -2, -4, -5, -6, -7)

(1, 0, -2, -4, -5, -6, -7)

(1, -1, -2, -3, -5, -6, -7)

(0, -1, -2, -3, -4, -6, -7)

[ 4  3  2  2  1  1  0]
[-2 -1  0 -1  0 -1 -1]
[-3 -3 -3 -2 -2 -2 -2]
[-4 -4 -4 -4 -4 -3 -3]
[-5 -5 -5 -5 -5 -5 -4]
[-6 -6 -6 -6 -6 -6 -6]
[-7 -7 -7 -7 -7 -7 -7]

In [139]:
M.inverse()

ZeroDivisionError: matrix must be nonsingular

## Fock vectors, partitions and swap distance

### References

[Lazy Power Series](http://doc.sagemath.org/html/en/reference/combinat/sage/combinat/tutorial.html)

[Lazy Laurent Series](http://doc.sagemath.org/html/en/reference/power_series/sage/rings/lazy_laurent_series.html)

[Integer Partitions](http://doc.sagemath.org/html/en/reference/combinat/sage/combinat/partition.html)


In [157]:
%display latex
import code as pc
import numpy as np
Partitions.options.latex='list'

In [158]:
part1 = [3,1,1]
fock_vec = [2, -1, -2]

In [160]:
def to_fock(partition):
    return vector([(x-(i+1)) for i, x in enumerate(part1)])

def to_partition(fock_vector):
    return Partition([(x+(i+1)) for i, x in enumerate(fock_vector)])

In [161]:
print(pc.fock(part1))
print(to_fock(part1))

[ 2 -1 -2]
(2, -1, -2)


In [162]:
print(pc.partition(fock_vec))
print(to_partition(fock_vec))

[3 1 1]
[3, 1, 1]


In [163]:
p = to_partition(fock_vec)
print(p)
print(p.ferrers_diagram())

[3, 1, 1]
***
*
*


In [164]:
Tableaux.options.ascii_art = "table"
print(p.hook_length(0, 0))
hlens = p.hook_lengths()
ascii_art(Tableau(hlens))

5


+---+---+---+
| 5 | 2 | 1 |
+---+---+---+
| 2 |
+---+
| 1 |
+---+

In [165]:
def alpha_k(v_lambda, k):
    M = Matrix([ v for i in range(v.length()) ])
    Av = M + identity_matrix(v.length()) * k
    return sum(Av)

In [166]:
v = to_fock(part1)
alpha_k(v, 2)

(8, -1, -4)

In [168]:
# Sum(A) for a matrix sums columns.
M = Matrix(QQ, 3, 
           [1,0,0, 
            2,0,1, 
            0,7,0

           ])
show(M)
sum(M)

[1 0 0]
[2 0 1]
[0 7 0]

(3, 7, 1)

In [169]:
fock = v
n = 1

M = Matrix([ fock for i in range(fock.length()) ])
Ik = identity_matrix(fock.length()) * n
Av = M - Ik
show(M, "+", Ik, "=", Av)

[ 2 -1 -2]
[ 2 -1 -2]
[ 2 -1 -2] '+' [1 0 0]
[0 1 0]
[0 0 1] '=' [ 1 -1 -2]
[ 2 -2 -2]
[ 2 -1 -3]

In [ ]:
Partitions.options(display="list")
for a in Partitions(5).list(): print(a)

In [172]:
Ms = pc.lowering1(v, 1)
show(v)
Ms

(2, -1, -2)

[array([[ 1, -1, -2],
        [ 2, -1, -3]]), [1, 1]]

In [176]:
fock = v
n = 1
s = len(fock)
# Stack vectors into square matrix
start = np.array([fock,]*s)
start

array([[ 2, -1, -2],
       [ 2, -1, -2],
       [ 2, -1, -2]])

In [177]:
for i in range(s):
    start[i][i]=start[i][i]-n
start

array([[ 1, -1, -2],
       [ 2, -2, -2],
       [ 2, -1, -3]])

In [178]:
# Pick rows where every entry is > -(1+s)
print(-1-s)
goodrows = start[np.all(start>-1-s, axis=1)]
goodrows

-4


array([[ 1, -1, -2],
       [ 2, -2, -2],
       [ 2, -1, -3]])

![](https://i.stack.imgur.com/h1alT.jpg)

In [179]:
betterrows = np.fliplr(np.sort(goodrows,axis=1))
betterrows

array([[ 1, -1, -2],
       [ 2, -2, -2],
       [ 2, -1, -3]])

In [ ]:
# Fliplr: puts columns in reverse order.
A = np.diag([1,2,3])
show(A)
B = np.fliplr(A)
show(B)

In [180]:
counterlist = []
norepeatlist = []
nottoobig = []
accuracy = 70

In [181]:
t=len(betterrows)
list(range(t))

[0, 1, 2]

In [182]:
goodrows

array([[ 1, -1, -2],
       [ 2, -2, -2],
       [ 2, -1, -3]])

In [186]:
i = 0
counter = -1
givenrow1 = goodrows[i]
givenrow1

array([ 1, -1, -2])

In [187]:
if sum(pc.partition(givenrow1))>accuracy:
    print("break")
givenrow2 = betterrows[i]
norepeats = True
givenrow2

array([ 1, -1, -2])

In [188]:
for j in range(s):
    if givenrow1[j]!=givenrow2[j]:
        counter = counter + 1
    if j<s-1:
        if givenrow2[j]==givenrow2[j+1]:
            norepeats = False
            break
show(counter)
show(norepeats)

-1

True

In [189]:
if norepeats==True:
    norepeatlist.append(i)
norepeatlist

[0]

In [190]:
counterlist.append(max(0,counter))
counterlist

[0]

In [192]:
counterlist = []
norepeatlist = []
nottoobig = []
t=len(betterrows)
for i in range(t):
    counter = -1
    givenrow1 = goodrows[i]
    if sum(pc.partition(givenrow1))>accuracy:
#            print('Warning: Set accuracy higher.')
        print("break")
    givenrow2 = betterrows[i]
    norepeats = True
    for j in range(s):
        if givenrow1[j]!=givenrow2[j]:
            counter = counter + 1
        if j<s-1:
            if givenrow2[j]==givenrow2[j+1]:
                norepeats = False
                break
    if norepeats==True:
        norepeatlist.append(i)
    counterlist.append(max(0,counter))

In [193]:
counterlist

[0, 0, 0]

In [194]:
norepeatlist

[0, 2]

### Swap Distance / Transposition Distance

Note: an NP-Hard problem [CS Paper from 2010](https://arxiv.org/abs/1011.1157)

Let $v, w$ be vectors in $\mathbb{Z}^n$ such that the entries of $v$ are in weakly decreasing order, and the entries of $w$ are some permutation of the entries of $v$.

Define $d(v, w)$ to be the number of transpositions needed to transform $v$ to $w$.

#### Algorithm 1 (Brute Force Count)
Perform a bubble sort and count the number of transpositions. Scales like $O(n^2)$ in the length of the vector.

#### Other Possibilities

**Algorithm 2**
Define a map
$$
S: \mathbb{Z} \to \mathbb{R}^{2n} \\
v \mapsto S(v)_{ij} = \begin{cases}1 & v_i < v_j \\ 0 & \text{else} \end{cases}
$$

Then (claim) $$d(v, w) = \| S(v) - S(w) \|_{2}^2.$$

[Reference: computing swap distances via Euclidean embedding](http://dimacs.rutgers.edu/~graham/pubs/slides/dmsa.pdf)

*Seems to be off by 1 sometimes..*.

**Matrix Rank:**
Compute the rank of an associated permutation matrix.
[Reference](https://math.berkeley.edu/~mhegde/other_math/permutation_distance.pdf)

Conjecture: Let $\sigma \in S^n$ be a permutation and $P(\sigma) \in \mathbb{Z}^{n^2}$ be its associated permutation matrix. Let $d(\sigma)$ be the number of transpositions that reduce $\sigma$ to the identity. Then

$$d(\sigma) = \mathrm{rank}(I_n - P(\sigma)).$$

Problem: how to convert an arbitrary vector to a permutation..?

**Direct Formula:**
Somehow convert vector of length $n$ to a cycle $\sigma \in S_n$. Then $d(\sigma, \mathrm{id}) = n - c(\sigma)$, the number of cycles in a disjoint cycle decomposition.

Problem: how to convert an arbitrary vector to a permutation..?


In [132]:
v = vector([1,2,1])
v

(1, 2, 1)

In [135]:
len(v) == len(set(v))

False

In [146]:
M = Matrix(QQ, 3, [1,1,1, 1,1,1, 1,2,3])
M

[1 1 1]
[1 1 1]
[1 2 3]

In [149]:
[R for R in M.rows() if len(R) == len(set(R))]

[(1, 2, 3)]

In [150]:
fock

NameError: name 'fock' is not defined